# Tutorial Básico de Gurobi para Programación Lineal

Este tutorial te enseñará los conceptos fundamentales para resolver problemas de programación lineal usando Gurobi en Python.

## Contenido:
1. Importar el módulo de Gurobi
2. Definir un problema ejemplo
3. Crear variables de decisión
4. Definir restricciones
5. Establecer la función objetivo
6. Resolver el problema
7. Extraer resultados y precios sombra


---

In [1]:
# Instalar la librería en caso de que no la tengas en tu dispositivo
!pip install gurobipy

## 1. Importación del Módulo Gurobi

Primero debemos importar la librería de Gurobi. Asegúrate de tener Gurobi instalado y con una licencia válida.

In [2]:
import gurobipy as gp
from gurobipy import GRB

# También importamos otra librerías útil para hacer DataFrames
import pandas as pd

## 2. Definición del Problema Ejemplo

Vamos a resolver un problema clásico de programación lineal:

**Problema de Producción:**

Una empresa produce dos productos: A y B. Queremos maximizar las ganancias.

- Producto A: ganancia de $3 por unidad
- Producto B: ganancia de $5 por unidad

**Restricciones:**
- Tiempo de máquina: 2 horas para A, 1 hora para B (máximo 100 horas disponibles)
- Material: 1 kg para A, 3 kg para B (máximo 80 kg disponibles)
- Demanda mínima: al menos 5 unidades de A
- Variables no negativas

**Formulación matemática:**

Maximizar: 3x₁ + 5x₂

Sujeto a:
- 2x₁ + x₂ ≤ 100  (tiempo de máquina)
- x₁ + 3x₂ ≤ 80   (material)
- x₁ ≥ 5          (demanda mínima)
- x₁, x₂ ≥ 0      (no negatividad)

## 3. Creación del Modelo

En Gurobi, todo problema se define dentro de un "modelo". Primero creamos el modelo:

In [3]:
# Crear el modelo
modelo = gp.Model("ProblemaProduccion")

Set parameter Username
Set parameter LicenseID to value 2817823
Academic license - for non-commercial use only - expires 2027-05-04


## 4. Definición de Variables de Decisión

Las variables de decisión representan las cantidades que queremos determinar. En nuestro caso, las cantidades a producir de cada producto.

Pese a ser positivas irrestrictas las variables, es buena práctica utilizar un número muy grande positivo como límite superior para mejorar la convergencia. Se denomina comúnmente `bigM` a este número muy grande. Para este ejemplo será 1 millón.

En este ejemplo las variables son continuas, por lo que su tipo será `GRB.CONTINUOUS`. Sin embargo, gurobi también permite definir otros tipos de variables, como binarias (`GRB.BINARY`) o enteras (`GRB.INTEGER`).

In [4]:
# Definir variables de decisión
# addVar(lb=lower_bound, ub=upper_bound, vtype=variable_type, name="nombre")

x1 = modelo.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoA") # Una variable puede ser Continuous, Binary o Integer
x2 = modelo.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoB")

# Actualizar el modelo para que reconozca las nuevas variables
modelo.update()

## 5. Definición de Restricciones

Las restricciones limitan el espacio de soluciones factibles. Se definen usando `addConstr()`.

In [5]:
# Definir restricciones

# Restricción 1: Tiempo de máquina (2x1 + x2 <= 100)
restriccion_tiempo = modelo.addConstr(2*x1 + x2 <= 100, name="TiempoMaquina")

# Restricción 2: Material (x1 + 3x2 <= 80)
restriccion_material = modelo.addConstr(x1 + 3*x2 <= 80, name="Material")

# Restricción 3: Demanda mínima (x1 >= 5)
restriccion_demanda = modelo.addConstr(x1 >= 5, name="DemandaMinima")

Las restricciones de no negatividad $(x1, x2 >= 0)$ ya están incluidas en la definición de las variables con $lb=0$

## 6. Definición de la Función Objetivo

La función objetivo define qué queremos optimizar (maximizar o minimizar).

In [6]:
# Definir función objetivo
# Maximizar: 3x1 + 5x2
modelo.setObjective(3*x1 + 5*x2, GRB.MAXIMIZE)

## 7. Resolución del Problema

Una vez definido completamente el modelo, usamos `optimize()` para resolverlo.

In [7]:
# Resolver el problema
modelo.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 5 nonzeros (Max)
Model fingerprint: 0x827fa6a8
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e+00, 5e+00]
  Bounds range     [1e+06, 1e+06]
  RHS range        [5e+00, 1e+02]

Presolve removed 1 rows and 0 columns
Presolve time: 0.01s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.6500000e+02   5.686200e+01   0.000000e+00      0s
       2    1.9200000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.920000000e+02


## 8. Extracción de Resultados

Si el problema tiene solución óptima, podemos extraer los valores de las variables y el valor objetivo.

In [8]:
# Extraer resultados si la solución es óptima
if modelo.status == GRB.OPTIMAL:
    print("=== RESULTADOS DE LA OPTIMIZACIÓN ===")
    print(f"Valor objetivo óptimo: ${modelo.objVal:.2f}")
    print()
    
    # Valores de las variables
    print("Valores óptimos de las variables:")
    print(f"Producto A (x1): {x1.x:.2f} unidades")
    print(f"Producto B (x2): {x2.x:.2f} unidades")
    print()
    
    # Verificar qué restricciones están activas (binding)
    print("Estado de las restricciones:")
    print(f"Tiempo de máquina: {2*x1.x + x2.x:.2f} / 100 horas")
    print(f"Material: {x1.x + 3*x2.x:.2f} / 80 kg")
    print(f"Demanda mínima: {x1.x:.2f} / 5 unidades mínimas")
else:
    print("No se pudo obtener una solución óptima")

=== RESULTADOS DE LA OPTIMIZACIÓN ===
Valor objetivo óptimo: $192.00

Valores óptimos de las variables:
Producto A (x1): 44.00 unidades
Producto B (x2): 12.00 unidades

Estado de las restricciones:
Tiempo de máquina: 100.00 / 100 horas
Material: 80.00 / 80 kg
Demanda mínima: 44.00 / 5 unidades mínimas


## 9. Extracción de Precios Sombra (Dual Values)

Los precios sombra nos indican cuánto aumentaría el valor objetivo si relajáramos cada restricción en una unidad.

In [9]:
# Extraer precios sombra (valores duales)
if modelo.status == GRB.OPTIMAL:
    print("=== PRECIOS SOMBRA (VALORES DUALES) ===")
    print()
    
    # Precios sombra de las restricciones
    print(f"Tiempo de máquina: ${restriccion_tiempo.pi:.2f} por hora adicional")
    print(f"Material: ${restriccion_material.pi:.2f} por kg adicional")
    print(f"Demanda mínima: ${restriccion_demanda.pi:.2f} por unidad adicional requerida")
    print()

=== PRECIOS SOMBRA (VALORES DUALES) ===

Tiempo de máquina: $0.80 por hora adicional
Material: $1.40 por kg adicional
Demanda mínima: $0.00 por unidad adicional requerida



# 10. Generación de problemas en formato matricial

Cuando los problemas están definidos mediante restricciones vectoriales, es más conveniente trabajar directamente con vectores. 
Por ejemplo:
$$
\begin{align}
&\max &&\mathbf{c}\cdot \mathbf{v}\\
&s.t. &&\mathbf{A} \mathbf{v} = \mathbf{b}\\
&&& \mathbf{D}\mathbf{v} \ge d\\
&&&\mathbf{v}_{LB} \leq \mathbf{v}\leq \mathbf{v}_{UB} 
\end{align}
$$
Este puede ser representado de la siguiente forma:

In [18]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

model_matrix = gp.Model('MatricialModel')

x = model_matrix.addMVar(shape=2,
                          lb=-10,
                          ub=10,
                          vtype=GRB.CONTINUOUS,
                          name="Productos")

# restricciones <=
A = np.array([[2, 1],
              [1, 3],
              [-1, 0]])
b = np.array([100, 80, 5])
model_matrix.addConstr(A @ x <= b, name="Restricciones_LE")

# nueva restricción >=
D = np.array([[1, 1]])   # ejemplo: x1 + x2 >= 10
d = np.array([10])
model_matrix.addConstr(D @ x >= d, name="Restricciones_GE")

c = np.array([3, 5])
model_matrix.setObjective(c @ x, GRB.MINIMIZE)

model_matrix.update()
model_matrix.optimize()

if model_matrix.status == GRB.OPTIMAL:
    print("Solución óptima:", x.X)
    print("Valor objetivo:", model_matrix.ObjVal)
else:
    print("Status:", model_matrix.status)


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 4 rows, 2 columns and 7 nonzeros (Min)
Model fingerprint: 0x874d4f37
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e+00, 5e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [5e+00, 1e+02]

Presolve removed 4 rows and 2 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.000000000e+01
Solución óptima: [10.  0.]
Valor objetivo: 30.0
